In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: Co2SiO4, D20

This example demonstrates a Rietveld refinement of Co2SiO4 crystal
structure using constant wavelength neutron powder diffraction data
from D20 at ILL.

It also shows different ways to set free parameters: standard
one-by-one and batch setting.

## 🛠️ Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## 🧩 Define Structure

This section shows how to add structures and modify their
parameters.

### Create Structure

In [3]:
structure = StructureFactory.from_scratch(name='cosio')

### Set Space Group

In [4]:
structure.space_group.name_h_m = 'P n m a'
structure.space_group.coord_system_code = 'abc'

### Set Unit Cell

In [5]:
structure.cell.length_a = 10.3
structure.cell.length_b = 6.0
structure.cell.length_c = 4.8

### Set Atom Sites

In [6]:
structure.atom_sites.create(
    id='Co1',
    type_symbol='Co',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='Co2',
    type_symbol='Co',
    fract_x=0.279,
    fract_y=0.25,
    fract_z=0.985,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='Si',
    type_symbol='Si',
    fract_x=0.094,
    fract_y=0.25,
    fract_z=0.429,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='O1',
    type_symbol='O',
    fract_x=0.091,
    fract_y=0.25,
    fract_z=0.771,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='O2',
    type_symbol='O',
    fract_x=0.448,
    fract_y=0.25,
    fract_z=0.217,
    adp_iso=0.5,
)
structure.atom_sites.create(
    id='O3',
    type_symbol='O',
    fract_x=0.164,
    fract_y=0.032,
    fract_z=0.28,
    adp_iso=0.5,
)

## 🔬 Define Experiment

This section shows how to add experiments, configure their parameters,
and link the structures defined in the previous step.

### Download Data

In [7]:
data_path = download_data('meas-cosio-d20', destination='data')

Getting data...


Data 'meas-cosio-d20': Co2SiO4, D20 (ILL)


✅ Data 'meas-cosio-d20' downloaded to '../../../data/meas-cosio-d20.xye'


### Create Experiment

In [8]:
expt = ExperimentFactory.from_data_path(name='d20', data_path=data_path)

### Set Instrument

In [9]:
expt.instrument.setup_wavelength = 1.87
expt.instrument.calib_twotheta_offset = 0.1

### Set Peak Profile

In [10]:
expt.peak.show_supported()

Peak types


,,Type,Description
1,*,pseudo-voigt,CWL pseudo-Voigt profile
2,,pseudo-voigt + berar-baldinozzi asymmetry,CWL pseudo-Voigt profile with Berar-Baldinozzi asymmetry correction.


In [11]:
expt.peak.type = 'pseudo-voigt + berar-baldinozzi asymmetry'

⚠️ Switching peak profile type adds these settings with defaults:                                                                 
   • asym_beba_a0=0.0                                                                                                             
   • asym_beba_a1=0.0                                                                                                             
   • asym_beba_b0=0.0                                                                                                             
   • asym_beba_b1=0.0                                                                                                             


Peak profile type for experiment 'd20' changed to


pseudo-voigt + berar-baldinozzi asymmetry


In [12]:
expt.peak.broad_gauss_u = 0.3
expt.peak.broad_gauss_v = -0.5
expt.peak.broad_gauss_w = 0.4

### Set Background

In [13]:
expt.background.show_supported()

Background types


,,Type,Description
1,,chebyshev,Chebyshev polynomial background
2,*,line-segment,Linear interpolation between points


In [14]:
expt.background.create(id='1', position=8, intensity=500)
expt.background.create(id='2', position=9, intensity=500)
expt.background.create(id='3', position=10, intensity=500)
expt.background.create(id='4', position=11, intensity=500)
expt.background.create(id='5', position=12, intensity=500)
expt.background.create(id='6', position=15, intensity=500)
expt.background.create(id='7', position=25, intensity=500)
expt.background.create(id='8', position=30, intensity=500)
expt.background.create(id='9', position=50, intensity=500)
expt.background.create(id='10', position=70, intensity=500)
expt.background.create(id='11', position=90, intensity=500)
expt.background.create(id='12', position=110, intensity=500)
expt.background.create(id='13', position=130, intensity=500)
expt.background.create(id='14', position=150, intensity=500)

### Set Linked Structures

In [15]:
expt.linked_structures.create(structure_id='cosio', scale=1.0)

## 📦 Define Project

The project object is used to manage the structure, experiment, and
analysis.

### Create Project

In [16]:
project = Project(name='cosio_d20')

In [17]:
project.save_as(dir_path='projects/refine-cosio-d20')

Saving project 📦 'cosio_d20' to '../../../projects/refine-cosio-d20'


├── 📄 project.edi


├── 📁 structures/


├── 📁 experiments/


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 cosio_d20.html


### Add Structure

In [18]:
project.structures.add(structure)

### Add Experiment

In [19]:
project.experiments.add(expt)

## 🚀 Perform Analysis

This section shows the analysis process, including how to set up
calculation and fitting engines.

### Display Structure

In [20]:
project.display.structure(struct_name='cosio')

Structure 🧩 'cosio' (Atom view type: 'covalent')


### Display Pattern

In [21]:
project.display.pattern(expt_name='d20')

In [22]:
project.display.pattern(expt_name='d20', x_min=41, x_max=54)

### Set Free Parameters

In [23]:
structure.cell.length_a.free = True
structure.cell.length_b.free = True
structure.cell.length_c.free = True

for atom_site in structure.atom_sites:
    for parameter in ('fract_x', 'fract_y', 'fract_z'):
        getattr(atom_site, parameter).free = True

for atom_site in structure.atom_sites:
    atom_site.adp_iso.free = True

for label in ('O1', 'O2', 'O3'):
    atom_site = structure.atom_sites[label]
    atom_site.occupancy.free = True

⚠️ Parameter 'cosio.atom_site.Co1.fract_x' is constrained by symmetry. Ignoring free=True.                                        


⚠️ Parameter 'cosio.atom_site.Co1.fract_y' is constrained by symmetry. Ignoring free=True.                                        


⚠️ Parameter 'cosio.atom_site.Co1.fract_z' is constrained by symmetry. Ignoring free=True.                                        


⚠️ Parameter 'cosio.atom_site.Co2.fract_y' is constrained by symmetry. Ignoring free=True.                                        


⚠️ Parameter 'cosio.atom_site.Si.fract_y' is constrained by symmetry. Ignoring free=True.                                         


⚠️ Parameter 'cosio.atom_site.O1.fract_y' is constrained by symmetry. Ignoring free=True.                                         


⚠️ Parameter 'cosio.atom_site.O2.fract_y' is constrained by symmetry. Ignoring free=True.                                         


In [24]:
expt.linked_structures['cosio'].scale.free = True

expt.instrument.calib_twotheta_offset.free = True

expt.peak.broad_gauss_u.free = True
expt.peak.broad_gauss_v.free = True
expt.peak.broad_gauss_w.free = True
expt.peak.broad_lorentz_y.free = True

expt.peak.asym_beba_b0.free = True

for point in expt.background:
    point.intensity.free = True

Show free parameters after selection.

In [25]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,cosio,cell,,length_a,10.30000,,-inf,inf,Å
2,cosio,cell,,length_b,6.00000,,-inf,inf,Å
3,cosio,cell,,length_c,4.80000,,-inf,inf,Å
4,cosio,atom_site,Co1,adp_iso,0.50000,,-inf,inf,Å²
5,cosio,atom_site,Co2,fract_x,0.27900,,-inf,inf,
6,cosio,atom_site,Co2,fract_z,0.98500,,-inf,inf,
7,cosio,atom_site,Co2,adp_iso,0.50000,,-inf,inf,Å²
8,cosio,atom_site,Si,fract_x,0.09400,,-inf,inf,
9,cosio,atom_site,Si,fract_z,0.42900,,-inf,inf,
10,cosio,atom_site,Si,adp_iso,0.50000,,-inf,inf,Å²


### Set Constraints

Set aliases for parameters.

In [26]:
project.analysis.aliases.create(
    id='biso_Co1',
    param=project.structures['cosio'].atom_sites['Co1'].adp_iso,
)
project.analysis.aliases.create(
    id='biso_Co2',
    param=project.structures['cosio'].atom_sites['Co2'].adp_iso,
)

Set constraints.

In [27]:
project.analysis.constraints.create(expression='biso_Co2 = biso_Co1')

### Run Fitting

In [28]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'd20' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

│   └── 📄 d20.edi


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 cosio_d20.html


In [29]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),51.94
4,🔁 Iterations,574
5,📏 Goodness-of-fit (reduced χ²),4.38
6,"📏 R-factor (Rf, %)",2.99
7,"📏 R-factor squared (Rf², %)",4.32
8,"📏 Weighted R-factor (wR, %)",4.54


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,cosio,cell,,length_a,Å,10.3000,10.3084,0.0003,0.08 % ↑
2,cosio,cell,,length_b,Å,6.0000,6.0036,0.0002,0.06 % ↑
3,cosio,cell,,length_c,Å,4.8000,4.7864,0.0001,0.28 % ↓
4,cosio,atom_site,Co1,adp_iso,Å²,0.5000,0.6716,0.1079,34.32 % ↑
5,cosio,atom_site,Co2,fract_x,,0.2790,0.2792,0.0007,0.06 % ↑
6,cosio,atom_site,Co2,fract_z,,0.9850,0.9850,0.0014,0.00 % ↓
7,cosio,atom_site,Si,fract_x,,0.0940,0.0938,0.0004,0.25 % ↓
8,cosio,atom_site,Si,fract_z,,0.4290,0.4293,0.0008,0.08 % ↑
9,cosio,atom_site,Si,adp_iso,Å²,0.5000,0.6856,0.0885,37.11 % ↑
10,cosio,atom_site,O1,fract_x,,0.0910,0.0911,0.0003,0.14 % ↑


In [30]:
project.display.fit.correlations()

### Display Pattern

In [31]:
project.display.pattern(expt_name='d20')

In [32]:
project.display.pattern(expt_name='d20', x_min=42, x_max=52)

## 📊 Report

The HTML report is written automatically when the project is saved;
enable `project.report.pdf` as well for a PDF version.